In [1]:
from google.colab import drive

drive.mount("/content/gdrive")

Mounted at /content/gdrive


TEMP-01 — Imports and paths

In [2]:
import os
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

TEMP_RESULTS_DIR = "/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Temporal"
os.makedirs(TEMP_RESULTS_DIR, exist_ok=True)

print("Temporal Analysis module initialized.")
print("Results directory:", TEMP_RESULTS_DIR)

Temporal Analysis module initialized.
Results directory: /content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Temporal


TEMP-02 — Define the temporal observation structure

In [3]:
temporal_observations = [
    {
        "observation_id": "OBS-001",
        "incident_id": "CW-001",
        "timestamp": "2026-09-11T10:00:00",
        "severity": "Moderate",
        "water_extent": 0.35,
        "people_affected": 20,
        "response_status": "Dispatched"
    },
    {
        "observation_id": "OBS-002",
        "incident_id": "CW-001",
        "timestamp": "2026-09-11T11:00:00",
        "severity": "High",
        "water_extent": 0.55,
        "people_affected": 45,
        "response_status": "En Route"
    },
    {
        "observation_id": "OBS-003",
        "incident_id": "CW-001",
        "timestamp": "2026-09-11T12:00:00",
        "severity": "High",
        "water_extent": 0.72,
        "people_affected": 80,
        "response_status": "Arrived"
    }
]

temporal_df = pd.DataFrame(temporal_observations)

temporal_df

,observation_id,incident_id,timestamp,severity,water_extent,people_affected,response_status
0,OBS-001,CW-001,2026-09-11T10:00:00,Moderate,0.35,20,Dispatched
1,OBS-002,CW-001,2026-09-11T11:00:00,High,0.55,45,En Route
2,OBS-003,CW-001,2026-09-11T12:00:00,High,0.72,80,Arrived


TEMP-03 — Convert timestamps and order observations

In [4]:
temporal_df["timestamp"] = pd.to_datetime(
    temporal_df["timestamp"]
)

temporal_df = temporal_df.sort_values(
    "timestamp"
).reset_index(drop=True)

print("Observations:", len(temporal_df))
print(
    "Time range:",
    temporal_df["timestamp"].min(),
    "to",
    temporal_df["timestamp"].max()
)

temporal_df

Observations: 3
Time range: 2026-09-11 10:00:00 to 2026-09-11 12:00:00


,observation_id,incident_id,timestamp,severity,water_extent,people_affected,response_status
0,OBS-001,CW-001,2026-09-11 10:00:00,Moderate,0.35,20,Dispatched
1,OBS-002,CW-001,2026-09-11 11:00:00,High,0.55,45,En Route
2,OBS-003,CW-001,2026-09-11 12:00:00,High,0.72,80,Arrived


TEMP-04 — Convert severity into an ordinal scale

In [5]:
severity_scale = {
    "Low": 1,
    "Moderate": 2,
    "High": 3
}

temporal_df["severity_score"] = temporal_df["severity"].map(
    severity_scale
)

temporal_df[[
    "observation_id",
    "timestamp",
    "severity",
    "severity_score"
]]

,observation_id,timestamp,severity,severity_score
0,OBS-001,2026-09-11 10:00:00,Moderate,2
1,OBS-002,2026-09-11 11:00:00,High,3
2,OBS-003,2026-09-11 12:00:00,High,3


TEMP-05 — Calculate temporal changes

In [6]:
temporal_df["severity_change"] = (
    temporal_df["severity_score"]
    .diff()
    .fillna(0)
)

temporal_df["water_extent_change"] = (
    temporal_df["water_extent"]
    .diff()
    .fillna(0)
)

temporal_df["people_affected_change"] = (
    temporal_df["people_affected"]
    .diff()
    .fillna(0)
)

temporal_df[[
    "observation_id",
    "timestamp",
    "severity_change",
    "water_extent_change",
    "people_affected_change"
]]

,observation_id,timestamp,severity_change,water_extent_change,people_affected_change
0,OBS-001,2026-09-11 10:00:00,0.0,0.00,0.0
1,OBS-002,2026-09-11 11:00:00,1.0,0.20,25.0
2,OBS-003,2026-09-11 12:00:00,0.0,0.17,35.0


TEMP-06 — Determine disaster trend

In [7]:
def determine_temporal_trend(row):
    severity_change = row["severity_change"]
    water_change = row["water_extent_change"]
    people_change = row["people_affected_change"]

    signals_increasing = 0
    signals_decreasing = 0

    if severity_change > 0:
        signals_increasing += 1
    elif severity_change < 0:
        signals_decreasing += 1

    if water_change > 0:
        signals_increasing += 1
    elif water_change < 0:
        signals_decreasing += 1

    if people_change > 0:
        signals_increasing += 1
    elif people_change < 0:
        signals_decreasing += 1

    if signals_increasing >= 2:
        return "INCREASING"

    if signals_decreasing >= 2:
        return "DECREASING"

    return "STABLE"


temporal_df["temporal_trend"] = temporal_df.apply(
    determine_temporal_trend,
    axis=1
)

temporal_df[[
    "observation_id",
    "timestamp",
    "severity",
    "water_extent",
    "people_affected",
    "temporal_trend"
]]

,observation_id,timestamp,severity,water_extent,people_affected,temporal_trend
0,OBS-001,2026-09-11 10:00:00,Moderate,0.35,20,STABLE
1,OBS-002,2026-09-11 11:00:00,High,0.55,45,INCREASING
2,OBS-003,2026-09-11 12:00:00,High,0.72,80,INCREASING


TEMP-07 — Save temporal result

In [8]:
temporal_csv_path = os.path.join(
    TEMP_RESULTS_DIR,
    "CrisisWeave_Temporal_Observations_CW001.csv"
)

temporal_df.to_csv(
    temporal_csv_path,
    index=False
)

print("Temporal observations saved:")
print(temporal_csv_path)

Temporal observations saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Temporal/CrisisWeave_Temporal_Observations_CW001.csv


TEMP-08 — Add spatial coordinates to temporal observations

In [9]:
# Prototype observation coordinates.
# These represent successive observations as the incident evolves.

temporal_df["latitude"] = [
    9.929117,
    9.930000,
    9.930850
]

temporal_df["longitude"] = [
    9.270436 if False else 76.270436,
    76.271200,
    76.272050
]

temporal_df[[
    "observation_id",
    "timestamp",
    "latitude",
    "longitude",
    "severity"
]]

,observation_id,timestamp,latitude,longitude,severity
0,OBS-001,2026-09-11 10:00:00,9.929117,76.270436,Moderate
1,OBS-002,2026-09-11 11:00:00,9.930000,76.271200,High
2,OBS-003,2026-09-11 12:00:00,9.930850,76.272050,High


TEMP-09 — Calculate movement between observations

In [10]:
from math import radians, sin, cos, sqrt, atan2

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371000  # metres

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1)
        * cos(lat2)
        * sin(dlon / 2) ** 2
    )

    return 2 * R * atan2(
        sqrt(a),
        sqrt(1 - a)
    )


distances = [0.0]

for i in range(1, len(temporal_df)):
    d = haversine_distance(
        temporal_df.loc[i-1, "latitude"],
        temporal_df.loc[i-1, "longitude"],
        temporal_df.loc[i, "latitude"],
        temporal_df.loc[i, "longitude"]
    )

    distances.append(d)

temporal_df["movement_from_previous_m"] = distances

temporal_df[[
    "observation_id",
    "timestamp",
    "movement_from_previous_m"
]]

,observation_id,timestamp,movement_from_previous_m
0,OBS-001,2026-09-11 10:00:00,0.000000
1,OBS-002,2026-09-11 11:00:00,129.006676
2,OBS-003,2026-09-11 12:00:00,132.667851


TEMP-10 — Estimate progression direction

In [11]:
def calculate_bearing(lat1, lon1, lat2, lon2):
    lat1 = radians(lat1)
    lat2 = radians(lat2)
    dlon = radians(lon2 - lon1)

    x = sin(dlon) * cos(lat2)

    y = (
        cos(lat1) * sin(lat2)
        - sin(lat1) * cos(lat2) * cos(dlon)
    )

    bearing = atan2(x, y)
    bearing = (bearing * 180 / np.pi + 360) % 360

    return bearing


bearings = [None]

for i in range(1, len(temporal_df)):
    bearing = calculate_bearing(
        temporal_df.loc[i-1, "latitude"],
        temporal_df.loc[i-1, "longitude"],
        temporal_df.loc[i, "latitude"],
        temporal_df.loc[i, "longitude"]
    )

    bearings.append(bearing)

temporal_df["progression_bearing"] = bearings

temporal_df[[
    "observation_id",
    "timestamp",
    "progression_bearing"
]]

,observation_id,timestamp,progression_bearing
0,OBS-001,2026-09-11 10:00:00,NaN
1,OBS-002,2026-09-11 11:00:00,40.439941
2,OBS-003,2026-09-11 12:00:00,44.567490


TEMP-11 — Convert bearing into direction

In [13]:
# TEMP-11 — Convert bearing into direction

def bearing_to_direction(bearing):
    # Handle both None and NaN
    if bearing is None or pd.isna(bearing):
        return "UNKNOWN"

    directions = [
        "N", "NE", "E", "SE",
        "S", "SW", "W", "NW"
    ]

    index = int((float(bearing) + 22.5) // 45) % 8

    return directions[index]


temporal_df["progression_direction"] = (
    temporal_df["progression_bearing"]
    .apply(bearing_to_direction)
)

print(
    temporal_df[
        [
            "observation_id",
            "timestamp",
            "progression_bearing",
            "progression_direction"
        ]
    ]
)

  observation_id           timestamp  progression_bearing  \
0        OBS-001 2026-09-11 10:00:00                  NaN   
1        OBS-002 2026-09-11 11:00:00            40.439941   
2        OBS-003 2026-09-11 12:00:00            44.567490   

  progression_direction  
0               UNKNOWN  
1                    NE  
2                    NE  


TEMP-12 — Build the temporal state

In [14]:
latest = temporal_df.iloc[-1]

temporal_state = {
    "incident_id": "CW-001",

    "observation_count": len(temporal_df),

    "first_observation": (
        temporal_df["timestamp"].min().isoformat()
    ),

    "latest_observation": (
        temporal_df["timestamp"].max().isoformat()
    ),

    "initial_severity": temporal_df.iloc[0]["severity"],

    "latest_severity": latest["severity"],

    "severity_trend": (
        "INCREASING"
        if temporal_df["severity_score"].iloc[-1]
        > temporal_df["severity_score"].iloc[0]
        else "STABLE"
    ),

    "water_extent_trend": (
        "INCREASING"
        if temporal_df["water_extent"].iloc[-1]
        > temporal_df["water_extent"].iloc[0]
        else "STABLE"
    ),

    "affected_population_trend": (
        "INCREASING"
        if temporal_df["people_affected"].iloc[-1]
        > temporal_df["people_affected"].iloc[0]
        else "STABLE"
    ),

    "overall_temporal_trend": (
        temporal_df["temporal_trend"].iloc[-1]
    ),

    "progression_direction": (
        temporal_df["progression_direction"].dropna().iloc[-1]
        if temporal_df["progression_direction"].notna().any()
        else "UNKNOWN"
    ),

    "latest_latitude": float(latest["latitude"]),
    "latest_longitude": float(latest["longitude"])
}

temporal_state

{'incident_id': 'CW-001',
 'observation_count': 3,
 'first_observation': '2026-09-11T10:00:00',
 'latest_observation': '2026-09-11T12:00:00',
 'initial_severity': 'Moderate',
 'latest_severity': 'High',
 'severity_trend': 'INCREASING',
 'water_extent_trend': 'INCREASING',
 'affected_population_trend': 'INCREASING',
 'overall_temporal_trend': 'INCREASING',
 'progression_direction': 'NE',
 'latest_latitude': 9.93085,
 'latest_longitude': 76.27205}

TEMP-13 — Save final temporal state

In [15]:
temporal_json_path = os.path.join(
    TEMP_RESULTS_DIR,
    "CrisisWeave_Temporal_State_CW001.json"
)

with open(
    temporal_json_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        temporal_state,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Temporal state saved:")
print(temporal_json_path)

Temporal state saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Temporal/CrisisWeave_Temporal_State_CW001.json


TEMP-14 — Integrate Temporal into CrisisWeave

In [17]:
# TEMP-14 — Create incident record and integrate Temporal information

crisis_incident = {
    "incident_id": "CW-001",

    "location": {
        "latitude": float(temporal_df.iloc[0]["latitude"]),
        "longitude": float(temporal_df.iloc[0]["longitude"]),
        "source": "user_map_pin"
    },

    "temporal": {
        "observation_count": temporal_state["observation_count"],
        "first_observation": temporal_state["first_observation"],
        "latest_observation": temporal_state["latest_observation"],
        "initial_severity": temporal_state["initial_severity"],
        "latest_severity": temporal_state["latest_severity"],
        "severity_trend": temporal_state["severity_trend"],
        "water_extent_trend": temporal_state["water_extent_trend"],
        "affected_population_trend": temporal_state["affected_population_trend"],
        "overall_temporal_trend": temporal_state["overall_temporal_trend"],
        "progression_direction": temporal_state["progression_direction"],
        "latest_latitude": temporal_state["latest_latitude"],
        "latest_longitude": temporal_state["latest_longitude"]
    }
}

print("Temporal information integrated successfully.\n")

print(json.dumps(
    crisis_incident["temporal"],
    indent=2,
    default=str
))

Temporal information integrated successfully.

{
  "observation_count": 3,
  "first_observation": "2026-09-11T10:00:00",
  "latest_observation": "2026-09-11T12:00:00",
  "initial_severity": "Moderate",
  "latest_severity": "High",
  "severity_trend": "INCREASING",
  "water_extent_trend": "INCREASING",
  "affected_population_trend": "INCREASING",
  "overall_temporal_trend": "INCREASING",
  "progression_direction": "NE",
  "latest_latitude": 9.93085,
  "latest_longitude": 76.27205
}


TEMP-15 — Save

In [18]:
# TEMP-15 — Save Temporal incident record

incident_path = (
    "/content/gdrive/MyDrive/"
    "CrisisWeave_Project/04_Results/"
    "Temporal/CrisisWeave_Incident_CW001_GIS_Temporal.json"
)

with open(incident_path, "w", encoding="utf-8") as f:
    json.dump(
        crisis_incident,
        f,
        indent=2,
        ensure_ascii=False,
        default=str
    )

print("Updated incident record saved:")
print(incident_path)

Updated incident record saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Temporal/CrisisWeave_Incident_CW001_GIS_Temporal.json
